# Aula 07 · Arquivos de texto e CSV

Este caderno é o [capítulo 7 do site](https://lacouth.github.io/python_telecom-site/unidade3-arquivos/07-arquivos-csv/) em forma de
aula: o mesmo texto, os mesmos exemplos, **sem as saídas**. Em cada exemplo:

1. **leia** o código, sem rodar;
2. **escreva** na célula `_Sua previsão:_`, logo abaixo dele, o que você acha
   que vai sair;
3. **rode** a célula do código e compare com o que você escreveu;
4. **abra** o `▶ O que aconteceu` para ler a explicação.

A previsão errada é a parte que ensina — não a apague.

**Ao fim desta aula você deve conseguir:**

1. ler um arquivo linha a linha com `with`, limpar cada linha e pular as vazias;
2. escolher entre `"w"` e `"a"` sabendo o que cada um faz com o conteúdo anterior;
3. ler um CSV com `DictReader`, guardar os registros numa lista e converter os campos antes de calcular.

## Parte 1 — O capítulo, exemplo a exemplo

Até aqui, os dados estavam escritos dentro do programa. Isso serviu para aprender
a lógica, mas nenhum trabalho real funciona assim: o log tem mil linhas, o export
de medições tem trinta mil, e ninguém vai colá-los dentro do código.

Este capítulo liga o que você já sabe fazer ao lugar de onde os dados vêm de
verdade.

!!! note "Os arquivos deste capítulo"
    Os exemplos usam dois arquivos pequenos, criados por este trecho:

In [ ]:
# Os exemplos deste capitulo leem arquivos. Este trecho cria os arquivos de
# exemplo na pasta de trabalho, para que o capitulo rode em qualquer maquina.
from pathlib import Path

PASTA = Path("trabalho")
PASTA.mkdir(exist_ok=True)

(PASTA / "alarmes.log").write_text(
    "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3\n"
    "2026-03-02 09:00:00 INFO SWITCH-NORTE-02 porta ativada\n"
    "2026-03-02 09:41:12 CRITICAL ONU-SUL-4512 sem resposta ha 12 minutos\n"
    "2026-03-02 10:02:55 CRITICAL OLT-CENTRO-01 temperatura acima do limite\n"
    "\n",
    encoding="utf-8",
)

(PASTA / "medicoes.csv").write_text(
    "equipamento,potencia_dbm,estado\n"
    "OLT-CENTRO-01,-21.4,UP\n"
    "ONU-SUL-4512,-27.0,UP\n"
    "OLT-NORTE-02,-19.8,DOWN\n",
    encoding="utf-8",
)

print("arquivos de exemplo criados em", PASTA.name)


<details>
<summary><b>▶ O que aconteceu</b></summary>

    Na lista de exercícios, a primeira célula do caderno faz o mesmo — assim tudo
    roda no Colab sem depender de download.

</details>

## Abrir, ler, fechar

In [ ]:
from pathlib import Path

CAMINHO = Path("trabalho") / "alarmes.log"

# 'with' abre, entrega o arquivo e FECHA sozinho ao sair do bloco.
with open(CAMINHO, encoding="utf-8") as arquivo:
    for linha in arquivo:
        print(repr(linha))


**Preveja:** "o arquivo tem 4 alarmes. Quantas linhas o laço vai imprimir?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Três coisas para reparar:

- **`with open(...) as arquivo:`** é a forma padrão. Ao sair do bloco, o arquivo é
  fechado — inclusive se der erro no meio. Abrir sem `with` funciona, mas exige
  lembrar do `.close()`, e é justamente o que ninguém lembra;
- **`encoding="utf-8"` sempre.** Sem isso, o Python usa a codificação padrão do
  sistema, que muda entre a sua máquina, o servidor e o Colab. O dia em que um
  nome tiver acento, o programa quebra num lugar só;
- **percorrer o arquivo com `for` entrega uma linha por vez**, sem carregar o
  arquivo inteiro na memória. É o que permite processar um log de 2 GB.

E o detalhe que o `repr` revelou: **cada linha vem com o `\n` no fim**. Se você
comparar `linha == "UP"` sem limpar, a comparação falha para sempre e você não vai
entender por quê.

</details>

## Limpando e contando

In [ ]:
from pathlib import Path

CAMINHO = Path("trabalho") / "alarmes.log"

criticos = 0
with open(CAMINHO, encoding="utf-8") as arquivo:
    for linha in arquivo:
        linha = linha.strip()
        if len(linha) == 0:        # a linha em branco do fim do arquivo
            continue
        if linha.split()[2] == "CRITICAL":
            criticos = criticos + 1

print(criticos)

# Lendo tudo de uma vez, quando o arquivo cabe na memoria.
with open(CAMINHO, encoding="utf-8") as arquivo:
    linhas = arquivo.read().splitlines()

print(len(linhas))
print(linhas[0].split()[3])


**Preveja:** "agora quantas linhas são contadas?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

O par de linhas que abre todo laço de leitura:

```python
linha = linha.strip()
if len(linha) == 0:
    continue
```

A primeira tira o `\n` e os espaços das pontas; a segunda **pula a linha vazia**.
Repare no resultado: o arquivo tem 4 alarmes, mas 5 linhas — a última é em branco,
porque todo arquivo de texto bem formado termina com uma quebra. Essa linha
fantasma é a causa número um de `IndexError` em script de log.

`arquivo.read().splitlines()` carrega tudo de uma vez numa lista, já sem os `\n`.
Use quando o arquivo é pequeno e você precisa percorrê-lo mais de uma vez.

</details>

## Escrever

In [ ]:
from pathlib import Path

PASTA = Path("trabalho")
SAIDA = PASTA / "relatorio.txt"

linhas = ["OLT-CENTRO-01     2", "ONU-SUL-4512      1"]

# "w" cria o arquivo, ou APAGA o conteudo do que ja existia.
with open(SAIDA, "w", encoding="utf-8") as arquivo:
    for linha in linhas:
        arquivo.write(linha + "\n")

print(SAIDA.read_text(encoding="utf-8"))

# "a" acrescenta no fim, sem apagar.
with open(SAIDA, "a", encoding="utf-8") as arquivo:
    arquivo.write("3 alarmes criticos\n")

print(SAIDA.read_text(encoding="utf-8"))


**Preveja:** "o que tem no arquivo depois das duas primeiras aberturas com `w`?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

| Modo | O que faz |
|---|---|
| `"r"` (padrão) | leitura |
| `"w"` | escrita — **apaga o conteúdo anterior** |
| `"a"` | acrescenta no fim, preservando o que havia |

O `"w"` apaga sem perguntar. Isso é conveniente para o relatório que se regenera
toda vez, e desastroso quando o nome do arquivo de saída coincide com o de
entrada. Confira duas vezes.

E `write` **não** acrescenta quebra de linha: o `+ "\n"` é seu.

</details>

## CSV

Um arquivo CSV é texto com campos separados por vírgula. Dá para lê-lo com
`split(",")` — e funciona, até o dia em que um campo contém uma vírgula dentro de
aspas. O módulo `csv` trata esses casos, e entrega cada linha já separada:

In [ ]:
import csv
from pathlib import Path

CAMINHO = Path("trabalho") / "medicoes.csv"

# csv.DictReader entrega uma LINHA POR VEZ, ja como dicionario: ele usa o
# cabecalho do arquivo como chave, e nao devolve o cabecalho como dado.
with open(CAMINHO, encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        print(registro)

# O acesso e por NOME de coluna, nao por posicao.
with open(CAMINHO, encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        print(registro["equipamento"], registro["estado"])


**Preveja:** "o arquivo tem 4 linhas. Quantas o laço imprime? E o cabeçalho, aparece?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`csv.DictReader` faz três coisas de uma vez:

- lê a **primeira linha do arquivo como cabeçalho**, e não a devolve como dado —
  por isso a saída tem três registros, e não quatro;
- transforma cada linha seguinte num **dicionário**, usando o cabeçalho como
  chave;
- entrega um registro por vez, como o `for` sobre o arquivo faz com as linhas.

O acesso é por nome de coluna: `registro["potencia_dbm"]`. Isso importa mais do
que parece — um `campos[3]` no meio do código não diz nada a quem lê, e quebra em
silêncio no dia em que alguém acrescentar uma coluna no meio do arquivo.

!!! tip "`newline=\"\"` ao abrir um CSV"
    É o que a documentação do módulo pede, e evita um problema de linhas em branco
    extras em arquivos gerados no Windows. Custa nada e um dia salva.

</details>

## Guardando as linhas para usar depois

O `DictReader` só funciona enquanto o arquivo está aberto, e passa **uma única
vez** pelas linhas. Quando você precisa dos dados depois — para percorrer duas
vezes, para calcular e depois filtrar —, acumule numa lista:

In [ ]:
import csv
from pathlib import Path

CAMINHO = Path("trabalho") / "medicoes.csv"

# O DictReader so funciona com o arquivo aberto, e so passa uma vez pelas linhas.
# Para guardar tudo e usar depois, acumule numa lista -- o laco de sempre.
medicoes = []
with open(CAMINHO, encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        medicoes.append(registro)

print(len(medicoes))
print(medicoes[0])
print(medicoes[0]["equipamento"])


**Preveja:** "que tipo de coisa é `medicoes` depois do laço?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

É o padrão de acumulação da Unidade 1, com dicionários no lugar de números. E
repare no que você tem agora: uma **lista de dicionários** — exatamente o formato
do capítulo 6.

A diferença entre os dois fica clara quando se percorre duas vezes:

</details>

In [ ]:
import csv
from pathlib import Path

CAMINHO = Path("trabalho") / "medicoes.csv"

# A lista responde quantas vezes voce perguntar...
medicoes = []
with open(CAMINHO, encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        medicoes.append(registro)
print(len(medicoes), len(medicoes))

# ...mas o DictReader passa UMA vez pelas linhas.
with open(CAMINHO, encoding="utf-8", newline="") as arquivo:
    leitor = csv.DictReader(arquivo)
    primeira = 0
    for registro in leitor:
        primeira = primeira + 1
    segunda = 0
    for registro in leitor:
        segunda = segunda + 1
print(primeira, segunda)


**Preveja:** "o segundo laço sobre o mesmo `DictReader` conta quantos registros?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A lista responde `3` quantas vezes você perguntar; o `DictReader` responde `3` e
depois `0`, porque ele é um leitor que avança e já chegou ao fim do arquivo.

</details>

## O CSV vira o que você já sabe processar

In [ ]:
import csv
from pathlib import Path

CAMINHO = Path("trabalho") / "medicoes.csv"

medicoes = []
with open(CAMINHO, encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        medicoes.append(registro)

# A partir daqui e a Unidade 2: uma lista de dicionarios, e os padroes de sempre.

# Acumulador -- e a conversao com float() e sua.
soma = 0.0
for medicao in medicoes:
    soma = soma + float(medicao["potencia_dbm"])
print(round(soma / len(medicoes), 2))

# Filtro
no_ar = []
for medicao in medicoes:
    if medicao["estado"] == "UP":
        no_ar.append(medicao["equipamento"])
print(no_ar)

# Contagem
por_estado = {}
for medicao in medicoes:
    estado = medicao["estado"]
    por_estado[estado] = por_estado.get(estado, 0) + 1
print(por_estado)


**Preveja:** "os três laços desta célula são padrões de qual aula?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Acumulador, filtro e contagem: nada aqui é novo. Só a origem do dado mudou.

E o aviso que vale para o curso inteiro: **tudo que vem do CSV é texto**, inclusive
o que parece número. `medicao["potencia_dbm"]` é a string `"-21.4"`, e sem o
`float(...)` a soma não é soma:

</details>

In [ ]:
import csv

# Um CSV minusculo, criado aqui mesmo para o exemplo rodar sozinho.
with open("medicoes_curtas.csv", "w", encoding="utf-8") as arquivo:
    arquivo.write("equipamento,potencia_dbm\nOLT-CENTRO-01,-21.4\nONU-SUL-4512,-27.0\n")

soma = 0
with open("medicoes_curtas.csv", encoding="utf-8", newline="") as arquivo:
    for registro in csv.DictReader(arquivo):
        soma = soma + registro["potencia_dbm"]   # faltou float()


**Preveja:** "por que esta soma falha, se a coluna tem números?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

## Quando o arquivo não está lá

</details>

In [ ]:
with open("coleta_de_ontem.csv", encoding="utf-8") as arquivo:
    print(arquivo.read())


**Preveja:** "qual é a mensagem quando o caminho está errado?"

_Sua previsão:_

→ escreva aqui e, só então, rode a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Reconheça a mensagem: ela é quase sempre um caminho errado, e não um arquivo
inexistente. No Colab, o arquivo criado pela primeira célula fica na pasta da
sessão — e **some quando o ambiente reinicia**. Se o erro aparecer do nada, rode a
célula de preparo de novo. Tratar esse erro sem derrubar o programa é assunto do
próximo capítulo.

</details>

## Onde isso é usado de verdade

Ler texto linha a linha é o que se faz com **syslog**, o fluxo pelo qual todo
equipamento reporta o que acontece com ele — e, em qualquer operação, o arquivo de
log é a primeira coisa que alguém abre quando o cliente reclama.

O CSV é o formato de troca do setor. Os contadores de desempenho de um elemento de
rede, o export de um sistema de gerência, a planilha de medições que o técnico
preenche em campo — tudo chega assim. E, na cadeia de faturamento, é a forma final
dos CDRs: os registros saem do núcleo da rede em
[formato binário ASN.1, especificado pelo 3GPP](https://docs.oracle.com/en/industries/communications/offline-mediation-controller/12.0/output-spec/wireless-output-formats.html),
passam pelo sistema de **mediação** — que decodifica, elimina duplicados e junta
registros parciais da mesma chamada — e é do outro lado dele que saem os arquivos
tabulares que as equipes de análise processam. O CSV que você lê aqui é o *depois*
da mediação, e é com ele que se trabalha na prática.

## Erros comuns deste capítulo

| Sintoma | Causa provável |
|---|---|
| `FileNotFoundError` | caminho errado, ou o arquivo ainda não foi criado nesta sessão do Colab |
| `IndexError` na última volta do laço | a linha em branco do fim do arquivo; use `if len(linha) == 0: continue` |
| comparação com string sempre falsa | faltou `.strip()`: a linha ainda tem o `\n` |
| `TypeError` ao somar valores do CSV | tudo que vem do arquivo é texto; converta com `float` ou `int` |
| acentos saem corrompidos | faltou `encoding="utf-8"` |
| o arquivo de saída ficou vazio | abriu com `"w"` e não escreveu, ou escreveu depois de fechar |
| o relatório apagou os dados de entrada | `"w"` no arquivo errado |

## Resumo

- `with open(caminho, encoding="utf-8") as f:` abre e fecha sozinho.
- O `for` sobre o arquivo entrega uma linha por vez, com `\n` no fim.
- `.strip()` e o `if len(linha) == 0: continue` abrem todo laço de leitura.
- `"w"` apaga, `"a"` acrescenta, `write` não põe quebra de linha.
- `csv.DictReader` lê o cabeçalho sozinho e entrega cada linha como dicionário.
- Acumulando esses registros numa lista você tem uma lista de dicionários, e a
  Unidade 2 inteira volta a valer.
- Todo campo lido de arquivo é **texto**.

_Anotações da Parte 1:_

## Parte 2 — Resolver junto

Tente sozinho primeiro, por cinco minutos, na sua máquina. Depois
resolvemos juntos.

### E1. Ler e limpar

Escreva `linhas_uteis(caminho)`, que abre o arquivo, devolve a lista das linhas
**sem a quebra de linha** e **sem as linhas em branco**.

```python
linhas_uteis("alarmes.log")[0]
# -> "2026-03-02 14:03:17 CRITICAL OLT-CENTRO-01 perda de sinal na porta GPON0/1/3"
len(linhas_uteis("alarmes.log"))   # -> 5
```

O arquivo tem seis linhas — a última é em branco, como em todo arquivo de texto
bem formado. Ela não deve entrar.

**Assinatura:**

```python
def linhas_uteis(caminho):
    """Linhas do arquivo, sem quebra de linha e sem as vazias."""
```

In [ ]:
# sua solução aqui


### E2. Contar críticos no arquivo

Escreva `conta_criticos(caminho)`, que devolve quantos alarmes de severidade
`CRITICAL` há no arquivo de log.

```python
conta_criticos("alarmes.log")  # -> 3
```

Cuidado com a linha em branco do fim: ela não tem campo nenhum, e tentar acessar
`campos[2]` nela levanta `IndexError`.

**Assinatura:**

```python
def conta_criticos(caminho):
    """Quantos alarmes CRITICAL há no arquivo."""
```

In [ ]:
# sua solução aqui


### E3. Gravar o relatório

Escreva `salva_relatorio(caminho, linhas)`, que grava cada item da lista numa
linha do arquivo e devolve **quantas linhas** gravou. O arquivo é sobrescrito a
cada chamada.

```python
salva_relatorio("saida.txt", ["OLT-A  2", "ONU-B  1"])   # -> 2
```

Lembre que `write` não acrescenta a quebra de linha — ela é sua.

**Assinatura:**

```python
def salva_relatorio(caminho, linhas):
    """Grava uma linha por item e devolve quantas foram gravadas."""
```

In [ ]:
# sua solução aqui


### E4. Média do CSV

Escreva `media_potencia(caminho)`, que devolve a média das potências do CSV,
arredondada para duas casas. Arquivo sem nenhuma medição devolve `0.0`.

```python
media_potencia("medicoes.csv")  # -> -24.07
```

Lembre: o campo lido do CSV é **texto**. A conversão é sua.

**Assinatura:**

```python
import csv

def media_potencia(caminho):
    """Média das potências do CSV, com duas casas. Arquivo vazio devolve 0.0."""
```

In [ ]:
# sua solução aqui


_Anotações da Parte 2:_

## Parte 3 — Quiz de conceitos

Responda de cabeça, sem rodar. Conferimos juntos no fim.

**1.** Um arquivo com 3 alarmes, gravado normalmente, tem quantas linhas percorridas pelo `for`?

a) 3  b) 4  c) 2  d) depende do sistema

**2.** `with open(caminho) as arquivo:` tem a vantagem de:

a) ser mais rápido  b) fechar o arquivo sozinho ao sair do bloco  c) ler o arquivo inteiro  d) converter os campos

**3.** Abrir com `"w"` um arquivo que já existe:

a) acrescenta no fim  b) dá erro  c) apaga o conteúdo anterior  d) cria um arquivo novo com outro nome

**4.** `csv.DictReader` usa como chaves:

a) os números das colunas  b) a primeira linha do arquivo  c) os nomes que você passar  d) as letras A, B, C

**5.** `registro["potencia_dbm"] + 1`, com o CSV lido normalmente:

a) soma 1 à potência  b) dá `TypeError`  c) concatena `"1"` no fim  d) devolve `None`